In [0]:
WITH validacao AS (

    SELECT
        'vendas' AS tabela,
        (SELECT COUNT(*)
         FROM jovi_database_aws_catalog.jovi_norte.vendas) AS origem,
        (SELECT COUNT(*)
         FROM jovi_lakehouse.bronze.vendas) AS bronze

    UNION ALL

    SELECT
        'promotores',
        (SELECT COUNT(*)
         FROM jovi_database_aws_catalog.jovi_norte.promotores),
        (SELECT COUNT(*)
         FROM jovi_lakehouse.bronze.promotores)

    UNION ALL

    SELECT
        'lojas',
        (SELECT COUNT(*)
         FROM jovi_database_aws_catalog.jovi_norte.lojas),
        (SELECT COUNT(*)
         FROM jovi_lakehouse.bronze.lojas)

    UNION ALL

    SELECT
        'produtos',
        (SELECT COUNT(*)
         FROM jovi_database_aws_catalog.jovi_norte.produtos),
        (SELECT COUNT(*)
         FROM jovi_lakehouse.bronze.produtos)

    UNION ALL

    SELECT
        'supervisores',
        (SELECT COUNT(*)
         FROM jovi_database_aws_catalog.jovi_norte.supervisores),
        (SELECT COUNT(*)
         FROM jovi_lakehouse.bronze.supervisores)

    UNION ALL

    SELECT
        'metas',
        (SELECT COUNT(*)
         FROM jovi_database_aws_catalog.jovi_norte.metas),
        (SELECT COUNT(*)
         FROM jovi_lakehouse.bronze.metas)
),

resultado AS (

    SELECT
        COUNT(*) AS qtd_tabelas,
        SUM(
            CASE
                WHEN origem = bronze THEN 0
                ELSE 1
            END
        ) AS qtd_erros,

        CONCAT_WS(
            ', ',
            COLLECT_LIST(
                CASE
                    WHEN origem <> bronze
                    THEN CONCAT(
                        tabela,
                        ' [AWS=',
                        origem,
                        ', BRONZE=',
                        bronze,
                        ']'
                    )
                END
            )
        ) AS detalhes_erros

    FROM validacao
)

SELECT
    assert_true(
        qtd_erros = 0,
        CONCAT(
            'DATA COMPLETENESS VALIDATION FAILED: ',
            detalhes_erros
        )
    )
FROM resultado;